# Ablation AB1: frame-consistent position embeddings (Cityscapes)

The training strategy is unchanged: SwinMTL's 512×1024 base and 448×896 crops, plus my random rescale [1, 1.5] and flip. Only the position embeddings of the crops change:
1. `geometric_aug` also returns the crop's window inside the rescaled frame (mirrored if flipped).
2. `DinoV2Backbone.frame_tokens` interpolates the table to the token grid of that **whole frame** (`ceil(frame/14)`, the same rounding as `pad_to_multiple` at test time) and slices the crop's window out of it.
3. At test time nothing changes: the full frame gets the standard interpolation, which is now exactly the grid seen during training.

Checked: with a full-frame window, `frame_tokens` gives exactly the standard tokens (max |diff| = 0).

**Everything else is identical to `cityscapes_partial_ft`** (B: 4 trainable blocks, critic on, same seed, 40k iterations), so this is a one-factor ablation. The run writes to `ckpt/fpe/` and `results/fpe/`.

## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
from mtl_train import *
from repro_common import PAPER   # the papers' reference numbers, defined once (checked against the PDFs)

configure()
print(f"Device: {device}, GPU: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'}")

AB1 = dict(frame_pe=True, ckpt_root=ROOT / "ckpt" / "fpe", results_root=ROOT / "results" / "fpe")


Device: cuda, GPU: NVIDIA RTX A4000


## Training

In [3]:
configure(**AB1)
hist_ab1 = train_run("cityscapes", "partial_ft")

cityscapes_partial_ft: already trained


## Evaluation
All Cityscapes protocols of notebook 03 (single pass, multi-scale + flip, sliding window) on the full val set, plus the per-image log bias. The results are compared with the main B run (`results/eval/cityscapes_partial_ft.json`) and with the paper numbers.

In [4]:
configure(**AB1)
out_path = CFG["results_root"] / "eval" / "cityscapes_partial_ft.json"
if out_path.exists():
    R_AB1 = load_json(out_path)
else:
    model = load_trained("cityscapes", "partial_ft")
    R_AB1 = eval_cityscapes(model, ms=True, slide=True)

    # mean log ratio prediction / GT at 512x1024 (the bias of notebook 03 §8.9)
    with torch.no_grad():
        dataset = make_dataset("cityscapes", "val", train=False)
        biases = []
        for i in range(len(dataset)):
            sample = dataset[i]
            gt = sample["depth"].to(device)
            gt = gt[0] if gt.ndim == 3 else gt
            depth = predict(model, sample["image"][None].to(device))[0]
            gt = F.interpolate(gt[None, None], size=depth.shape[-2:], mode="nearest")[0, 0]
            valid = (gt > 1e-3) & (gt < 80)
            biases.append((torch.log(depth[0, 0][valid]) - torch.log(gt[valid])).mean().item())
    R_AB1["log_bias"] = float(np.mean(biases))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    save_json(R_AB1, out_path)
    del model
    torch.cuda.empty_cache()

configure()
R_B = load_json(ROOT / "results" / "eval" / "cityscapes_partial_ft.json")

rows = [
    {"model": "SwinMTL (Tab. I / II)", "AbsRel": PAPER["swinmtl_tab1"]["SwinMTL"]["abs_rel"], "SqRel": PAPER["swinmtl_tab1"]["SwinMTL"]["sq_rel"], "RMSE (512)": PAPER["swinmtl_tab1"]["SwinMTL"]["rmse"], "RMSElog": PAPER["swinmtl_tab1"]["SwinMTL"]["rmse_log"], "δ1": PAPER["swinmtl_tab1"]["SwinMTL"]["d1"], "RMSE (full)": PAPER["swinmtl_tab2"]["SwinMTL"][0], "mIoU (paper: as reported; ours: standard)": PAPER["swinmtl_tab2"]["SwinMTL"][1], "mIoU +ms (TTA)": None},
    {"model": "DINOv2 ViT-S/14 (Tab. 10, frozen: lin. / +ms)", "mIoU (paper: as reported; ours: standard)": PAPER["dinov2_tab10_vits14"]["cityscapes"][0], "mIoU +ms (TTA)": PAPER["dinov2_tab10_vits14"]["cityscapes"][1]},
]
for label, R in [("B (main run)", R_B), ("AB1: B + frame-consistent PE", R_AB1)]:
    for suffix, d512, dfull, seg in [("", "cs_depth_512x1024", "cs_depth_full", "cs_seg"),
                                      (" +slide (TTA)", "cs_depth_512x1024_slide", "cs_depth_full_slide", "cs_seg_slide")]:
        if d512 not in R:
            continue
        rows.append({
            "model": label + suffix, "AbsRel": R[d512]["abs_rel"], "SqRel": R[d512]["sq_rel"], "RMSE (512)": R[d512]["rmse"],
            "RMSElog": R[d512]["rmse_log"], "δ1": R[d512]["d1"], "RMSE (full)": R[dfull]["rmse"],
            "mIoU (paper: as reported; ours: standard)": R[seg]["mIoU"], "mIoU +ms (TTA)": R["cs_seg_ms"]["mIoU"] if suffix == "" else None,
        })
ab1_table = pd.DataFrame(rows).set_index("model")
ab1_table.to_csv(ROOT / "results" / "compare" / "ablation_ab1_cityscapes.csv")
display(ab1_table.style.format(lambda v: "—" if pd.isna(v) else f"{v:.4f}" if v < 2 else f"{v:.3f}" if v < 10 else f"{v:.2f}"))
print(f"log bias (single pass, 512x1024): AB1 {R_AB1.get('log_bias', float('nan')):+.4f}   (B: +0.0747, recomputed in ablations.ipynb)")

,AbsRel,SqRel,RMSE (512),RMSElog,δ1,RMSE (full),mIoU (paper: as reported; ours: standard),mIoU +ms (TTA)
model,,,,,,,,
SwinMTL (Tab. I / II),0.0890,1.0510,5.481,0.1390,0.9210,6.352,76.41,—
"DINOv2 ViT-S/14 (Tab. 10, frozen: lin. / +ms)",—,—,—,—,—,—,66.60,77.10
B (main run),0.1221,0.9793,5.423,0.1577,0.8968,5.453,77.26,78.81
B (main run) +slide (TTA),0.0893,0.7572,4.860,0.1363,0.9197,4.893,79.21,—
AB1: B + frame-consistent PE,0.1077,0.8673,5.184,0.1492,0.9062,5.212,76.99,78.75
AB1: B + frame-consistent PE +slide (TTA),0.0891,0.7439,4.847,0.1368,0.9179,4.879,78.49,—


log bias (single pass, 512x1024): AB1 +0.0522   (B: +0.0747, recomputed in ablations.ipynb)
